# Chương 12: Thực hành - Mô hình Bộ biến đổi (Transformer Architecture from Scratch)

**Mục tiêu bài thực hành:**
1. **Cơ chế Scaled Dot-Product Attention:** Tự cài đặt từ đầu bằng `NumPy` và kiểm chứng lý do toán học vì sao bắt buộc phải chia cho $\sqrt{d_k}$.
2. **Multi-Head Attention từ đầu:** Xây dựng cơ chế chú ý đa đầu với phép phân tách và ghép nối các chiều biểu diễn.
3. **Mã hóa Vị trí Hình sin (Sinusoidal Positional Encoding):** Lập trình công thức giải tích của Vaswani et al. và trực quan hóa bản đồ nhiệt cùng tính chất khoảng cách tương đối.
4. **Mặt nạ Nhân quả (Causal Masking):** Cài đặt ma trận che giấu tương lai ứng dụng trong các mô hình sinh tự hồi quy (GPT Decoder).
5. **Mô phỏng Patch Embedding của Vision Transformer (ViT):** Biến đổi ảnh 2D thành chuỗi vector nhúng kèm token `[CLS]`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Thiết lập hạt giống ngẫu nhiên
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Cơ chế Scaled Dot-Product Attention và Tác dụng của $\sqrt{d_k}$

$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{Softmax}\left( \frac{\mathbf{Q} \mathbf{K}^T}{\sqrt{d_k}} + M \right) \mathbf{V}$$

In [ ]:
def softmax(z, axis=-1):
    exp_z = np.exp(z - np.max(z, axis=axis, keepdims=True))
    return exp_z / np.sum(exp_z, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None, scale=True):
    d_k = Q.shape[-1]
    # Tính tích vô hướng Q * K^T
    scores = Q @ K.T
    if scale:
        scores = scores / np.sqrt(d_k)
        
    if mask is not None:
        # Áp dụng mặt nạ: cộng -1e9 vào các vị trí bị che
        scores = scores + (mask * -1e9)
        
    attn_weights = softmax(scores, axis=-1)
    output = attn_weights @ V
    return output, attn_weights

# Thử nghiệm khi d_k lớn (d_k = 64)
seq_len = 5
d_k = 64
Q_sample = np.random.randn(seq_len, d_k)
K_sample = np.random.randn(seq_len, d_k)
V_sample = np.random.randn(seq_len, d_k)

_, attn_scaled = scaled_dot_product_attention(Q_sample, K_sample, V_sample, scale=True)
_, attn_unscaled = scaled_dot_product_attention(Q_sample, K_sample, V_sample, scale=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
im1 = ax1.imshow(attn_scaled, cmap='Blues', aspect='auto')
ax1.set_title("Có chia √d_k (Scaled)\nPhân phối xác suất mềm mại, gradient khỏe", fontsize=11, fontweight='bold')
plt.colorbar(im1, ax=ax1)

im2 = ax2.imshow(attn_unscaled, cmap='Blues', aspect='auto')
ax2.set_title("Không chia √d_k (Unscaled)\nBão hòa cực đoan (0 hoặc 1), gradient triệt tiêu", fontsize=11, fontweight='bold')
plt.colorbar(im2, ax=ax2)

plt.suptitle("Kiểm chứng Thực nghiệm Tác dụng của Hệ số Tỷ lệ √d_k trong Self-Attention", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 2. Cài đặt Multi-Head Attention (MHA from scratch)

In [ ]:
class MultiHeadAttentionScratch:
    def __init__(self, d_model=16, n_heads=4):
        assert d_model % n_heads == 0, "d_model phải chia hết cho n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        # Khởi tạo các ma trận trọng số chiếu
        self.W_q = np.random.randn(d_model, d_model) * 0.1
        self.W_k = np.random.randn(d_model, d_model) * 0.1
        self.W_v = np.random.randn(d_model, d_model) * 0.1
        self.W_o = np.random.randn(d_model, d_model) * 0.1
        
    def forward(self, X, mask=None):
        N = X.shape[0] # Chiều dài chuỗi token
        
        # Chiếu tuyến tính: N x d_model
        Q_all = X @ self.W_q
        K_all = X @ self.W_k
        V_all = X @ self.W_v
        
        # Tách thành n_heads đầu chú ý độc lập
        head_outputs = []
        for h in range(self.n_heads):
            # Lấy lát cắt vector chiều d_k cho head thứ h
            q_h = Q_all[:, h*self.d_k : (h+1)*self.d_k]
            k_h = K_all[:, h*self.d_k : (h+1)*self.d_k]
            v_h = V_all[:, h*self.d_k : (h+1)*self.d_k]
            
            out_h, _ = scaled_dot_product_attention(q_h, k_h, v_h, mask=mask)
            head_outputs.append(out_h)
            
        # Ghép nối (Concat) tất cả các heads lại
        concat_heads = np.hstack(head_outputs) # N x d_model
        final_output = concat_heads @ self.W_o
        return final_output

mha = MultiHeadAttentionScratch(d_model=16, n_heads=4)
X_tokens = np.random.randn(6, 16) # Chuỗi 6 từ, mỗi từ 16 chiều
output_mha = mha.forward(X_tokens)
print(f"Đầu vào: {X_tokens.shape} -> Đầu ra qua Multi-Head Attention: {output_mha.shape}")

## 3. Mã hóa Vị trí Hình sin (Sinusoidal Positional Encoding)

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right), \qquad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

In [ ]:
def get_sinusoidal_positional_encoding(seq_len=60, d_model=64):
    pe = np.zeros((seq_len, d_model))
    pos = np.arange(seq_len)[:, np.newaxis] # seq_len x 1
    i = np.arange(0, d_model, 2)[np.newaxis, :] # 1 x (d_model/2)
    
    angle_rates = 1.0 / np.power(10000.0, (2 * (i // 2)) / float(d_model))
    angles = pos * angle_rates
    
    pe[:, 0::2] = np.sin(angles)
    pe[:, 1::2] = np.cos(angles)
    return pe

PE_matrix = get_sinusoidal_positional_encoding(seq_len=60, d_model=64)

plt.figure(figsize=(12, 5))
plt.imshow(PE_matrix, cmap='viridis', aspect='auto')
plt.colorbar(label='Giá trị sóng sin/cos')
plt.title("Bản đồ nhiệt Ma trận Mã hóa Vị trí (Sinusoidal Positional Encoding)", fontsize=13, fontweight='bold')
plt.xlabel("Chỉ số Chiều nhúng (Embedding Dimensions d=64)")
plt.ylabel("Vị trí Token trong câu (Positions 0 -> 59)")
plt.tight_layout()
plt.show()

## 4. Mặt nạ Nhân quả (Causal Masking / Look-Ahead Mask trong GPT Decoder)

Trong bài toán sinh văn bản, từ ở vị trí $t$ tuyệt đối không được nhìn thấy các từ ở tương lai $t+1, t+2$.

In [ ]:
seq_len_demo = 6
# Tạo ma trận mặt nạ tam giác trên (1 ở các vị trí tương lai cần che)
causal_mask = np.triu(np.ones((seq_len_demo, seq_len_demo)), k=1)

Q_dec = np.random.randn(seq_len_demo, 8)
K_dec = np.random.randn(seq_len_demo, 8)
V_dec = np.random.randn(seq_len_demo, 8)

_, attn_causal = scaled_dot_product_attention(Q_dec, K_dec, V_dec, mask=causal_mask)

tokens = ["Tôi", "đang", "học", "mô", "hình", "Transformer"]
plt.figure(figsize=(7, 6))
plt.imshow(attn_causal, cmap='Blues')
plt.colorbar(label='Trọng số Chú ý (Attention Weight)')
plt.xticks(range(seq_len_demo), tokens, fontsize=10)
plt.yticks(range(seq_len_demo), tokens, fontsize=10)
plt.title("Ma trận Mặt nạ Nhân quả Causal Masking\n(Tam giác trên bị triệt tiêu hoàn toàn về 0)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Tổng kết bài học thực hành

1. **Scaled Dot-Product Attention:** Hệ số $\frac{1}{\sqrt{d_k}}$ đóng vai trò sinh tử giữ cho gradient không bị triệt tiêu khi chiều nhúng lớn.
2. **Multi-Head Attention:** Cho phép mạng đồng thời khai thác nhiều mối quan hệ ngữ nghĩa phân cấp trong một bước tính toán duy nhất.
3. **Positional Encoding:** Bù đắp tính vô hướng của phép toán Attention, giúp mô hình nắm bắt trật tự cú pháp của ngôn ngữ.
4. **Causal Mask:** Nền móng kỹ thuật của toàn bộ các mô hình sinh ngôn ngữ tự hồi quy (GPT-3/4, LLaMA).